# C7-cnn-transfer — Session 4: CNN Training and Selective Fine-Tuning

*One 90-minute session. Prerequisites: C6's `nn.Module` and parameter
inspection, C11's stable cross-entropy/autograd/optimizer lifecycle, and C7
Sessions 1–3's convolution, shape tracing, truncation, and freezing.*

**Learning contract.** We will build one tiny convolutional classifier, trace
its exact shape flow into a flattened head, train it on deterministic synthetic
images, and certify the run using loss, optimizer ownership, gradients,
parameter movement, and BatchNorm buffers. We then freeze before constructing
an optimizer, fine-tune selectively, and rebuild the optimizer after an
intentional unfreeze. Everything is CPU-small, seed `20260804`, and uses no
pretrained weights, downloads, or network access.

In [ ]:
from copy import deepcopy

import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
torch.manual_seed(20260804)
SEED = 20260804
ATOL = 1e-10
RTOL = 1e-8


## 1. From a dense trainer to a convolutional classifier

C11 trained logits shaped `(N,C)` with cross-entropy. That contract does not
change. Only the feature extractor changes: an image begins as `(N,C_in,H,W)`,
convolutions preserve spatial structure, and a flattening boundary converts
the final feature grid to one vector per example.

For input `(N,1,8,8)`, our exact flow is:

| operation | output shape |
|---|---|
| `Conv2d(1,4,3,padding=1)` | `(N,4,8,8)` |
| `BatchNorm2d(4)`, ReLU | `(N,4,8,8)` |
| `MaxPool2d(2)` | `(N,4,4,4)` |
| `Conv2d(4,6,3,padding=1)` | `(N,6,4,4)` |
| `BatchNorm2d(6)`, ReLU | `(N,6,4,4)` |
| `AdaptiveAvgPool2d((2,2))` | `(N,6,2,2)` |
| `Flatten(1)` | `(N,24)` |
| dropout, `Linear(24,3)` | `(N,3)` |

The head's `24` is derived, not guessed: $6\cdot2\cdot2$.
`CrossEntropyLoss` consumes raw logits and integer targets; do **not** put a
softmax in the model.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(4)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(4, 6, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(6)
        self.adapt = nn.AdaptiveAvgPool2d((2, 2))
        self.dropout = nn.Dropout(p=0.10)
        self.head = nn.Linear(6 * 2 * 2, 3)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.adapt(x)
        x = torch.flatten(x, 1)
        return self.head(self.dropout(x))


def shape_ledger(model, x):
    rows = []
    hooks = []
    for name in ("conv1", "bn1", "pool", "conv2", "bn2", "adapt", "head"):
        module = getattr(model, name)
        hooks.append(module.register_forward_hook(
            lambda _m, _i, out, name=name: rows.append((name, tuple(out.shape)))
        ))
    model.eval()
    with torch.no_grad():
        logits = model(x)
    for hook in hooks:
        hook.remove()
    return rows, logits

shape_model = TinyCNN()
shape_rows, shape_logits = shape_ledger(shape_model, torch.zeros(5, 1, 8, 8))
assert shape_rows == [
    ("conv1", (5, 4, 8, 8)), ("bn1", (5, 4, 8, 8)),
    ("pool", (5, 4, 4, 4)), ("conv2", (5, 6, 4, 4)),
    ("bn2", (5, 6, 4, 4)), ("adapt", (5, 6, 2, 2)),
    ("head", (5, 3)),
]
assert shape_logits.shape == (5, 3)


### Checkpoint 1

1. Why is the flatten width 24 rather than 96?
2. If the adaptive pool becomes `(1,1)`, what must change in the head?
3. Why is a softmax layer before `CrossEntropyLoss` a contract error?

## 2. Deterministic synthetic images and a complete training step

Each class is a noisy geometric pattern: a vertical bar, horizontal bar, or
diagonal. The generator order is fixed, and the model seed is reset before
construction. Fixed seeds are necessary but the **draw order** and training
order are part of reproducibility too.

One step has exactly five state transitions:

```text
zero_grad(set_to_none=True) → forward → CrossEntropyLoss → backward → step
```

`backward()` constructs/accumulates gradients; only `step()` moves optimizer-
owned parameters. BatchNorm buffers can move during the forward even though
they are not parameters and are absent from the optimizer.

In [ ]:
def make_images(seed=SEED, repeats=8):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    images = []
    labels = []
    for label in range(3):
        for _ in range(repeats):
            image = 0.04 * torch.randn(1, 8, 8, generator=generator)
            if label == 0:
                image[:, :, 2:4] += 1.0
            elif label == 1:
                image[:, 4:6, :] += 1.0
            else:
                image[:, torch.arange(8), torch.arange(8)] += 1.0
            images.append(image)
            labels.append(label)
    return torch.stack(images), torch.tensor(labels, dtype=torch.long)

X_train, y_train = make_images()
assert X_train.shape == (24, 1, 8, 8)
assert y_train.shape == (24,)


In [ ]:
torch.manual_seed(SEED)
model = TinyCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.20, momentum=0.0)
initial_parameters = {name: p.detach().clone() for name, p in model.named_parameters()}
initial_buffers = {name: b.detach().clone() for name, b in model.named_buffers()}
losses = []
last_grad_names = []
model.train()
for _ in range(30):
    optimizer.zero_grad(set_to_none=True)
    logits = model(X_train)
    loss = criterion(logits, y_train)
    losses.append(float(loss.detach()))
    loss.backward()
    last_grad_names = [name for name, p in model.named_parameters() if p.grad is not None]
    optimizer.step()

assert torch.isfinite(torch.tensor(losses)).all()
assert losses[-1] < 0.35 * losses[0]
assert set(last_grad_names) == {name for name, _ in model.named_parameters()}
assert any(
    not torch.equal(parameter.detach(), initial_parameters[name])
    for name, parameter in model.named_parameters()
)
assert any(
    not torch.equal(buffer.detach(), initial_buffers[name])
    for name, buffer in model.named_buffers()
    if "running_" in name
)


### Worked audit

Suppose loss falls but `conv1.weight` never moves. Four different causes can
look alike until separated:

1. it is intentionally frozen (`requires_grad=False`);
2. it is absent from the optimizer;
3. the graph is disconnected (`grad is None`); or
4. its gradient is exactly zero for this batch.

The repair is not “train longer.” Audit the flag, optimizer object identity,
gradient state/norm, and before/after snapshot in that order. Likewise, a
changed `bn1.running_mean` is **buffer** movement caused by a training-mode
forward, not an optimizer update.

### Checkpoint 2

1. Which line first creates parameter gradients, and which line moves weights?
2. Why can BatchNorm buffers change before `backward()`?
3. Why is “final loss is finite” weaker than the certificate above?

## 3. Optimizer ownership and freeze-before-optimizer

An optimizer captures parameter objects at construction. The intended set is
therefore an identity-level contract:

```python
owned = {id(p) for group in optimizer.param_groups for p in group["params"]}
expected = {id(p) for p in model.parameters() if p.requires_grad}
assert owned == expected
```

Freeze first, construct second. Passing frozen parameters to an optimizer may
not move them today when their gradients stay `None`, but it violates explicit
ownership and makes later state changes error-prone.

In [ ]:
fine_tune = deepcopy(model)
for name, parameter in fine_tune.named_parameters():
    parameter.requires_grad = not (name.startswith("conv1") or name.startswith("bn1"))

fine_optimizer = torch.optim.SGD(
    [p for p in fine_tune.parameters() if p.requires_grad], lr=0.05
)
owned_ids = {id(p) for group in fine_optimizer.param_groups for p in group["params"]}
expected_ids = {id(p) for p in fine_tune.parameters() if p.requires_grad}
assert owned_ids == expected_ids
assert all(id(p) not in owned_ids for p in fine_tune.parameters() if not p.requires_grad)


## 4. Selective fine-tuning certificate

We now fine-tune on a slightly shifted synthetic batch. A robust certificate
does not demand exact final weights. It requires:

- finite, meaningfully lower cross-entropy;
- optimizer ownership exactly equal to the trainable parameter identities;
- gradients on every expected trainable parameter and none on frozen ones;
- at least one allowed parameter moves;
- every frozen parameter remains **bitwise** unchanged; and
- running buffers are reported separately from parameters.

Training mode is intentional here: BatchNorm buffers may adapt even when its
affine parameters are frozen. If buffer immobility is required, put those
modules in evaluation mode explicitly; parameter freezing alone does not do
that.

In [ ]:
X_shift, y_shift = make_images(seed=SEED + 1, repeats=6)
frozen_before = {
    name: p.detach().clone() for name, p in fine_tune.named_parameters()
    if not p.requires_grad
}
trainable_before = {
    name: p.detach().clone() for name, p in fine_tune.named_parameters()
    if p.requires_grad
}
buffers_before = {name: b.detach().clone() for name, b in fine_tune.named_buffers()}
fine_losses = []
fine_tune.train()
for _ in range(12):
    fine_optimizer.zero_grad(set_to_none=True)
    logits = fine_tune(X_shift)
    loss = criterion(logits, y_shift)
    fine_losses.append(float(loss.detach()))
    loss.backward()
    fine_optimizer.step()

gradient_names = {name for name, p in fine_tune.named_parameters() if p.grad is not None}
expected_gradient_names = {name for name, p in fine_tune.named_parameters() if p.requires_grad}
assert gradient_names == expected_gradient_names
assert fine_losses[-1] < fine_losses[0]
assert all(torch.equal(p.detach(), frozen_before[name])
           for name, p in fine_tune.named_parameters() if not p.requires_grad)
assert any(not torch.equal(p.detach(), trainable_before[name])
           for name, p in fine_tune.named_parameters() if p.requires_grad)
training_buffers_moved = any(
    not torch.equal(buffer.detach(), buffers_before[name])
    for name, buffer in fine_tune.named_buffers() if "running_" in name
)
assert training_buffers_moved


## 5. Evaluation and selective unfreezing

Evaluation is a separate state transition. Call `eval()` so dropout becomes
identity and BatchNorm reads rather than updates buffers; use `no_grad()` so no
graph is created. Snapshot parameters **and buffers** before the pass.

To unfreeze `conv1`, flip its flag and then rebuild the optimizer. An existing
optimizer does not discover newly trainable parameters automatically.

In [ ]:
eval_parameters = {name: p.detach().clone() for name, p in fine_tune.named_parameters()}
eval_buffers = {name: b.detach().clone() for name, b in fine_tune.named_buffers()}
fine_tune.eval()
with torch.no_grad():
    eval_logits_1 = fine_tune(X_shift)
    eval_logits_2 = fine_tune(X_shift)
assert not eval_logits_1.requires_grad
assert torch.allclose(eval_logits_1, eval_logits_2, atol=ATOL, rtol=RTOL)
assert all(torch.equal(p.detach(), eval_parameters[name])
           for name, p in fine_tune.named_parameters())
assert all(torch.equal(b.detach(), eval_buffers[name])
           for name, b in fine_tune.named_buffers())

for name, parameter in fine_tune.named_parameters():
    if name.startswith("conv1"):
        parameter.requires_grad = True
rebuilt_optimizer = torch.optim.SGD(
    [p for p in fine_tune.parameters() if p.requires_grad], lr=0.02
)
rebuilt_owned = {id(p) for group in rebuilt_optimizer.param_groups for p in group["params"]}
assert rebuilt_owned == {id(p) for p in fine_tune.parameters() if p.requires_grad}


### Checkpoint 3

1. Why does changing `requires_grad` after optimizer construction not change
   optimizer membership?
2. Can frozen BatchNorm affine parameters coexist with moving running buffers?
3. What two independent calls define deterministic evaluation behavior?

## 6. Common pitfalls and exam connections

- **Guessed flatten width:** derive every spatial axis, then multiply; adaptive
  pooling makes the head independent of the incoming grid.
- **Softmax before CE:** `CrossEntropyLoss` expects logits and performs a stable
  log-softmax internally.
- **Wrong lifecycle:** stale gradients or a cleared-before-step gradient can
  produce plausible losses with wrong updates.
- **Freeze after optimizer construction:** flags and ownership disagree.
- **Mode confusion:** `eval()` changes BatchNorm/dropout behavior but does not
  disable gradients; `no_grad()` disables graph recording but does not select
  evaluation behavior.
- **Loss-only evidence:** report ownership, gradients, movement, and buffers as
  independent invariants.

Round 1 items often ask for one missing lifecycle line, an exact output shape,
or which state can change under a control. The capstones `p10`, `p24`, `p26`,
and `p27` combine those short registers into full construction audits.

## Checkpoint answers

**1.1** Pooling changes `8×8` to `4×4`, then adaptive pooling changes it to
`2×2`; the six channels therefore flatten to `6·2·2=24`, not `6·4·4=96`.
**1.2** The head's input width becomes `6·1·1=6`.
**1.3** CE expects logits and performs the stable fused log-softmax; feeding
probabilities changes the objective and its gradients.

**2.1** `loss.backward()` constructs/accumulates gradients;
`optimizer.step()` moves owned parameters.
**2.2** BatchNorm's forward logic updates running buffers in training mode;
the optimizer and backward pass do not own them.
**2.3** A finite loss can stay flat or rise, while disconnected parameters
and ownership mistakes remain invisible.

**3.1** Optimizers store the parameter objects supplied at construction; they
do not rescan the model.
**3.2** Yes. Affine parameters obey autograd/optimizer controls, while buffers
obey BatchNorm's mode-dependent forward rule.
**3.3** `model.eval()` plus a no-gradient context (`torch.no_grad()` or
`torch.inference_mode()`).